# ECG-FM Stage 2 — Kaggle One-Click

### Before running:

**1. Add datasets** (Edit → Settings → Add-ons, or the + icon in the Data section):
- `shlomihazan/ptbxl-500hz` — PTB-XL 500 Hz signals
- `shlomihazan/ecgfm-models` — model weights + training scripts

**2. Enable GPU:** Edit → Settings → Accelerator → GPU T4 x2

**3. Run the single cell below** — setup, training, and results all in one.

---
Output model `ecgfm_stage2.pt` will appear in the **Output** tab when training finishes (~3 hrs).

In [ ]:
# ── Single cell: setup + train + show results ────────────────────────────────
import os, shutil, zipfile, torch, psutil
from pathlib import Path

# ── 1. Check GPU ─────────────────────────────────────────────────────────────
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU — enable T4 GPU: Edit → Settings → Accelerator')
print('RAM:', round(psutil.virtual_memory().total / 1e9, 1), 'GB\n')

# ── 2. Auto-detect input datasets ────────────────────────────────────────────
PTBXL_PATH = None
MODEL_PATH  = None

print('Available datasets:')
for d in sorted(os.listdir('/kaggle/input')):
    print(f'  /kaggle/input/{d}/')
    candidate = f'/kaggle/input/{d}'
    if os.path.exists(f'{candidate}/ptbxl_database.csv'):
        PTBXL_PATH = candidate
    if os.path.exists(f'{candidate}/ecgfm_stage1.pt'):
        MODEL_PATH = candidate
print()

if not PTBXL_PATH:
    raise FileNotFoundError('PTB-XL not found — add shlomihazan/ptbxl-500hz via the Data section.')
if not MODEL_PATH:
    raise FileNotFoundError('ecgfm-models not found — add shlomihazan/ecgfm-models via the Data section.')

for fname in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py',
              'mimic_iv_ecg_physionet_pretrained.pt', 'ecgfm_stage1.pt']:
    if not os.path.exists(f'{MODEL_PATH}/{fname}'):
        raise FileNotFoundError(f'Missing {fname} in ecgfm-models dataset.')

print(f'PTB-XL path  : {PTBXL_PATH}')
print(f'Model path   : {MODEL_PATH}\n')

# ── 3. Set up working directory ───────────────────────────────────────────────
WORKDIR = '/kaggle/working/ecgfm'
os.makedirs(f'{WORKDIR}/models/ecgfm', exist_ok=True)
os.makedirs(f'{WORKDIR}/ekg_datasets', exist_ok=True)

# Copy scripts
for script in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py']:
    shutil.copy(f'{MODEL_PATH}/{script}', f'{WORKDIR}/{script}')

# Copy model weights
shutil.copy(f'{MODEL_PATH}/mimic_iv_ecg_physionet_pretrained.pt',
            f'{WORKDIR}/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_PATH}/ecgfm_stage1.pt', f'{WORKDIR}/models/ecgfm_stage1.pt')

# Extract or symlink PTB-XL
ptbxl_link = f'{WORKDIR}/ekg_datasets/ptbxl'
if os.path.islink(ptbxl_link):
    os.remove(ptbxl_link)

zip_path = f'{PTBXL_PATH}/records500.zip'
if os.path.exists(zip_path):
    # Extract the zip into the ptbxl folder
    os.makedirs(ptbxl_link, exist_ok=True)
    print('Extracting records500.zip (this takes ~2 min)...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ptbxl_link)
    shutil.copy(f'{PTBXL_PATH}/ptbxl_database.csv', f'{ptbxl_link}/ptbxl_database.csv')
    shutil.copy(f'{PTBXL_PATH}/scp_statements.csv',  f'{ptbxl_link}/scp_statements.csv')
    n_dat = len(list(Path(ptbxl_link).rglob('*.dat')))
    print(f'Extracted: {n_dat} signal files')
else:
    # Direct symlink (fallback if dataset has unzipped records500/)
    os.symlink(os.path.abspath(PTBXL_PATH), ptbxl_link)
    n_dat = len(list(Path(ptbxl_link).rglob('*.dat')))
    print(f'PTB-XL symlinked: {n_dat} signal files')

if n_dat < 20000:
    raise RuntimeError(f'Only {n_dat}/21837 signal files found — check the ptbxl-500hz dataset.')

print(f'\nSetup complete:')
print(f'  Scripts    : ecgfm_finetune.py, ecgfm_encoder.py, cnn_classifier.py')
print(f'  Encoder    : {round(os.path.getsize(WORKDIR+"/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt")/1e6)} MB')
print(f'  Stage1 ckpt: {round(os.path.getsize(WORKDIR+"/models/ecgfm_stage1.pt")/1e6)} MB')
print(f'  PTB-XL     : {n_dat} signal files\n')

# ── 4. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'scipy', 'scikit-learn'],
               check=True)
print('Dependencies ready\n')

# ── 5. Run Stage 2 training ───────────────────────────────────────────────────
# ~20 min/epoch on T4, ~3 hours total with early stopping (patience=8)
# Checkpoint saved to models/ecgfm_stage2.pt at every validation improvement
# If CUDA OOM: change --batch_s2 8 to --batch_s2 4
os.chdir(WORKDIR)
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

# ── 6. Results ────────────────────────────────────────────────────────────────
model_path = f'{WORKDIR}/models/ecgfm_stage2.pt'
if os.path.exists(model_path):
    print(f'\necgfm_stage2.pt saved ({round(os.path.getsize(model_path)/1e6)} MB)')
    print('Download: Output tab → ecgfm/models/ecgfm_stage2.pt\n')
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print('Final test results:')
    print(f"  Accuracy : {tm.get('acc',  0):.1%}")
    print(f"  HYP F1   : {tm.get('hyp_f1',   0):.3f}  (baseline 0.375)")
    print(f"  Macro F1 : {tm.get('macro_f1', 0):.3f}  (baseline 0.492)")
else:
    print('WARNING: ecgfm_stage2.pt not found — check the cell output above for errors')